## 1. Extração — Coleta via API CoinGecko (top 250 moedas)
Consulta o endpoint `/coins/markets` da CoinGecko para buscar os dados de mercado das 250 principais criptomoedas (em USD), usando a chave de API carregada do arquivo `.env`. O resultado é armazenado como JSON em `dados_json`.

In [2]:
import requests
import os
from dotenv import load_dotenv

load_dotenv()

base_url = "https://api.coingecko.com/api/v3/coins/markets"
parametros = {
    "vs_currency": "usd",
    "order": "market_cap_desc",
    "per_page": 250,
    "page": 1,
    "x_cg_demo_api_key": os.getenv("COINGECKO_API_KEY")
}

response = requests.get(base_url, params=parametros)
print(response.status_code)
dados_json = response.json()
print(len(dados_json))  # deve mostrar 250

200
250


## 2. Transformação — Seleção das colunas de interesse
Converte o JSON retornado pela API em um DataFrame do pandas e mantém apenas as colunas relevantes para a análise (preço, market cap, volume, variação de 24h, ATH etc.), descartando o restante do payload.

In [3]:
import pandas as pd

df_cripto = pd.DataFrame(dados_json)

colunas_interesse = [
    "id", "symbol", "name", "current_price", "market_cap", "market_cap_rank",
    "total_volume", "high_24h", "low_24h", "price_change_percentage_24h",
    "ath", "ath_change_percentage"
]

df_cripto = df_cripto[colunas_interesse]
df_cripto.head()

,id,symbol,name,current_price,market_cap,market_cap_rank,total_volume,high_24h,low_24h,price_change_percentage_24h,ath,ath_change_percentage
0,bitcoin,btc,Bitcoin,64333.000000,1290592300053,1,1.452050e+10,64384.00000,63761.000000,0.2,126080.000,-48.97477
1,ethereum,eth,Ethereum,1874.260000,226190682646,2,4.576842e+09,1875.84000,1851.520000,0.7,4946.050,-62.10590
2,tether,usdt,Tether,0.999293,184026670127,3,2.578565e+10,0.99931,0.999017,0.0,1.320,-24.47311
3,binancecoin,bnb,BNB,568.990000,75769820067,4,3.759034e+08,569.24000,563.520000,0.7,1369.990,-58.46770
4,usd-coin,usdc,USDC,0.999812,72541755690,5,5.755452e+09,1.00000,0.999568,0.0,1.043,-4.18347


## 3. Carga — Persistência da camada Bronze
Adiciona o timestamp `data_coleta`, abre a conexão com o PostgreSQL (credenciais do `.env`) e grava o DataFrame na tabela `bronze_cripto`, substituindo o conteúdo existente a cada execução — dados brutos, sem tratamento.

In [4]:
import pandas as pd
from sqlalchemy import create_engine

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

df_cripto["data_coleta"] = pd.Timestamp.now()
df_cripto.to_sql("bronze_cripto", engine, if_exists="replace", index=False)
print("Carga concluída na tabela bronze_cripto")

Carga concluída na tabela bronze_cripto


## 4. Camada Silver — Limpeza, padronização e engenharia de atributos
Lê os dados brutos de `bronze_cripto`, remove duplicatas e linhas sem informação essencial, padroniza tipos e textos, e cria métricas derivadas (`amplitude_pct_24h`, `posicao_no_range_24h`, `volume_milhoes`, `categoria_ath`). Em seguida grava o resultado na tabela `silver_cripto`.

In [13]:
df_bronze = pd.read_sql("SELECT * FROM bronze_cripto", engine)

df_silver = df_bronze.copy()

# Remove duplicatas por moeda, mantendo o registro mais recente
df_silver = df_silver.drop_duplicates(subset="id", keep="last")

# Remove linhas sem informação essencial
df_silver = df_silver.dropna(subset=["id", "current_price", "market_cap"])

# Padroniza tipos das colunas numéricas
colunas_float = [
    "current_price", "market_cap", "total_volume", "high_24h", "low_24h",
    "price_change_percentage_24h", "ath", "ath_change_percentage"
]
df_silver[colunas_float] = df_silver[colunas_float].astype(float)
df_silver["market_cap_rank"] = df_silver["market_cap_rank"].astype("Int64")

# Padroniza texto
df_silver["id"] = df_silver["id"].str.strip().str.lower()
df_silver["symbol"] = df_silver["symbol"].str.strip().str.lower()
df_silver["name"] = df_silver["name"].str.strip()

# Timestamp da coleta, para manter histórico entre execuções
df_silver["data_coleta"] = pd.Timestamp.now()

# Métricas da camada Silver
df_silver["amplitude_pct_24h"] = ((df_silver["high_24h"] - df_silver["low_24h"]) / df_silver["low_24h"]) * 100
df_silver["posicao_no_range_24h"] = (df_silver["current_price"] - df_silver["low_24h"]) / (df_silver["high_24h"] - df_silver["low_24h"])
df_silver["volume_milhoes"] = df_silver["total_volume"] / 1_000_000
df_silver["categoria_ath"] = pd.cut(
    df_silver["ath_change_percentage"],
    bins=[-100.01, -75, -50, -25, 0],
    labels=["Muito longe do topo", "Longe do topo", "Perto do topo", "Próximo do topo"]
)

df_silver.head()

,id,symbol,name,current_price,market_cap,market_cap_rank,total_volume,high_24h,low_24h,price_change_percentage_24h,ath,ath_change_percentage,data_coleta,amplitude_pct_24h,posicao_no_range_24h,volume_milhoes,categoria_ath
0,bitcoin,btc,Bitcoin,64333.000000,1.290592e+12,1,1.452050e+10,64384.00000,63761.000000,0.2,126080.000,-48.97477,2026-07-25 20:21:48.217192,0.977086,0.918138,14520.500809,Perto do topo
1,ethereum,eth,Ethereum,1874.260000,2.261907e+11,2,4.576842e+09,1875.84000,1851.520000,0.7,4946.050,-62.10590,2026-07-25 20:21:48.217192,1.313515,0.935033,4576.841742,Longe do topo
2,tether,usdt,Tether,0.999293,1.840267e+11,3,2.578565e+10,0.99931,0.999017,0.0,1.320,-24.47311,2026-07-25 20:21:48.217192,0.029329,0.941980,25785.651231,Próximo do topo
3,binancecoin,bnb,BNB,568.990000,7.576982e+10,4,3.759034e+08,569.24000,563.520000,0.7,1369.990,-58.46770,2026-07-25 20:21:48.217192,1.015048,0.956294,375.903397,Longe do topo
4,usd-coin,usdc,USDC,0.999812,7.254176e+10,5,5.755452e+09,1.00000,0.999568,0.0,1.043,-4.18347,2026-07-25 20:21:48.217192,0.043219,0.564815,5755.452285,Próximo do topo


In [14]:
df_silver.to_sql("silver_cripto", engine, if_exists="replace", index=False)
print(f"Carga concluída na tabela silver_cripto ({len(df_silver)} linhas)")

Carga concluída na tabela silver_cripto (250 linhas)


## Gold 1 — Volatilidade (ranking)
Ordena as moedas pela amplitude percentual entre máxima e mínima nas últimas 24h (`amplitude_pct_24h`), destacando também em que ponto do range diário o preço atual está (`posicao_no_range_24h`), usado para identificar as moedas mais voláteis do snapshot.

In [16]:
gold_volatilidade = df_silver[["id", "name", "amplitude_pct_24h", "posicao_no_range_24h"]] \
    .sort_values("amplitude_pct_24h", ascending=False) \
    .reset_index(drop=True)

gold_volatilidade.head(10)

,id,name,amplitude_pct_24h,posicao_no_range_24h
0,dexe,DeXe,86.005831,0.532203
1,zama,Zama,30.247276,0.450572
2,lorenzo-protocol,Lorenzo Protocol,24.868839,0.669961
3,quack-ai,Quack AI,24.035981,1.025496
4,shiba-inu,Shiba Inu,23.021583,0.687500
5,velvet,Velvet,16.733487,0.690547
6,audiera,Audiera,16.442953,0.775510
7,sosovalue,SoSoValue,14.006383,0.717254
8,build-on,BUILDon,14.000359,0.050684
9,bitway,Bitway,13.145155,1.311360


## Gold 2 — Distância do all-time high
Ordena as moedas pela distância percentual em relação ao seu recorde histórico de preço (`ath_change_percentage`) e reaproveita a classificação `categoria_ath` da camada Silver para destacar quais ativos estão mais próximos ou mais distantes do topo.

In [17]:
gold_distancia_ath = df_silver[["id", "name", "ath_change_percentage", "categoria_ath"]] \
    .sort_values("ath_change_percentage") \
    .reset_index(drop=True)

gold_distancia_ath.head(10)

,id,name,ath_change_percentage,categoria_ath
0,tradable-na-rent-financing-platform-sstn,Tradable NA Rent Financing Platform SSTN,-100.00000,Muito longe do topo
1,terra-luna,Terra Luna Classic,-99.99995,Muito longe do topo
2,apyusd,apyUSD,-99.97509,Muito longe do topo
3,sun-token,Sun Token,-99.97245,Muito longe do topo
4,jasmycoin,JasmyCoin,-99.90914,Muito longe do topo
5,ai-analysis-token,AI Analysis Token,-99.71686,Muito longe do topo
6,internet-computer,Internet Computer,-99.69268,Muito longe do topo
7,filecoin,Filecoin,-99.68773,Muito longe do topo
8,apecoin,ApeCoin,-99.46303,Muito longe do topo
9,the-sandbox,The Sandbox,-99.46142,Muito longe do topo


## Gold 3 — Concentração de mercado (nova)
Calcula qual fatia do market cap total das 250 moedas coletadas pertence às 10 maiores (`concentracao_top10_pct`). Como a tabela é gravada com `append`, cada execução acumula um novo registro, permitindo acompanhar a evolução da concentração ao longo do tempo.

In [18]:
market_cap_total = df_silver["market_cap"].sum()
top10_market_cap = df_silver.sort_values("market_cap", ascending=False).head(10)["market_cap"].sum()

concentracao_pct = (top10_market_cap / market_cap_total) * 100

gold_concentracao = pd.DataFrame({
    "data_coleta": [df_silver["data_coleta"].iloc[0]],
    "market_cap_total_250": [market_cap_total],
    "market_cap_top10": [top10_market_cap],
    "concentracao_top10_pct": [concentracao_pct]
})

gold_concentracao

,data_coleta,market_cap_total_250,market_cap_top10,concentracao_top10_pct
0,2026-07-25 20:21:48.217192,2.273987e+12,2.030279e+12,89.282787


## Gold 4 — Correlação com Bitcoin (nova)
Compara a variação percentual de 24h de cada moeda com a do Bitcoin no mesmo snapshot, ordenando pela menor diferença absoluta (`diferenca_variacao_vs_bitcoin`) para identificar quais ativos se moveram de forma mais parecida com o BTC.

In [19]:
preco_bitcoin_var = df_silver.loc[df_silver["id"] == "bitcoin", "price_change_percentage_24h"].values[0]

# Correlação entre a variação do bitcoin (constante nesse snapshot) não funciona linha a linha
# Em vez disso, calculamos a correlação entre TODAS as moedas e o bitcoin ao longo de múltiplos snapshots futuros
# Por ora (snapshot único), mostramos a diferença de cada moeda em relação ao movimento do bitcoin:

df_silver["diferenca_variacao_vs_bitcoin"] = df_silver["price_change_percentage_24h"] - preco_bitcoin_var

gold_correlacao_bitcoin = df_silver[["id", "name", "price_change_percentage_24h", "diferenca_variacao_vs_bitcoin"]] \
    .sort_values("diferenca_variacao_vs_bitcoin", key=abs) \
    .reset_index(drop=True)

gold_correlacao_bitcoin.head(10)

,id,name,price_change_percentage_24h,diferenca_variacao_vs_bitcoin
0,bitcoin,Bitcoin,0.2,0.0
1,sui,Sui,0.2,0.0
2,pax-gold,PAX Gold,0.2,0.0
3,tether-gold,Tether Gold,0.2,0.0
4,celestia,Celestia,0.2,0.0
5,reallink,RealLink,0.2,0.0
6,decred,Decred,0.2,0.0
7,stellar,Stellar,0.3,0.1
8,loaded-lions,Loaded Lions,0.3,0.1
9,gnosis,Gnosis,0.3,0.1


## Load das 4 tabelas Gold
Grava as quatro tabelas Gold no PostgreSQL: `gold_volatilidade`, `gold_distancia_ath` e `gold_correlacao_bitcoin` são substituídas (`replace`) a cada execução, enquanto `gold_concentracao_mercado` é acumulada (`append`) para preservar o histórico.

In [20]:
gold_volatilidade.to_sql("gold_volatilidade", engine, if_exists="replace", index=False)
gold_distancia_ath.to_sql("gold_distancia_ath", engine, if_exists="replace", index=False)
gold_concentracao.to_sql("gold_concentracao_mercado", engine, if_exists="append", index=False)
gold_correlacao_bitcoin.to_sql("gold_correlacao_bitcoin", engine, if_exists="replace", index=False)

print("Carga concluída nas 4 tabelas Gold")

Carga concluída nas 4 tabelas Gold
